# KvForge ProLAD — WikiText In-Domain Benchmark

Trains ProLAD (cosine) + Baseline (immediate) on WikiText-103.
Evaluates PPL(off) and PPL(on) at 64→4096 tokens.


In [ ]:
import json, math, time, gc, random
import torch
import torch.nn as nn
import torch.nn.functional as F

print("=" * 70)
print("KvForge ProLAD — WikiText In-Domain Long-Context Benchmark")
print("=" * 70)

device = "cuda" if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 7 else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"  Device: {device} | Dtype: {dtype}")

# ==== Model ====
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
print(f"\nLoading {MODEL}...")
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=dtype).to(device).eval()
print(f"  {sum(p.numel() for p in base.parameters())/1e6:.1f}M params")

# ==== LoRA ====
class LoRALinear(nn.Module):
    def __init__(self, orig, r=8, alpha=16):
        super().__init__()
        self.orig = orig
        self.scaling = alpha / r
        dt = orig.weight.dtype
        self.lora_A = nn.Parameter(torch.randn(orig.in_features, r, dtype=dt) * 0.02)
        self.lora_B = nn.Parameter(torch.zeros(r, orig.out_features, dtype=dt))
        self.active = True
    def activate(self, a=True): self.active = a
    def forward(self, x):
        h = self.orig(x)
        if self.active:
            h = h + (x @ self.lora_A @ self.lora_B) * self.scaling
        return h

def inject_lora(model, r=8):
    count = 0
    for n, m in model.named_modules():
        if any(n.endswith(s) for s in [".q_proj", ".k_proj", ".v_proj", ".o_proj"]):
            if isinstance(m, nn.Linear) and not isinstance(m, LoRALinear):
                parent = model; parts = n.split(".")
                for p in parts[:-1]:
                    if p: parent = getattr(parent, p)
                setattr(parent, parts[-1], LoRALinear(m, r=r))
                count += 1
    return count

def set_lora(model, active):
    for mod in model.modules():
        if hasattr(mod, "activate"):
            mod.activate(active)

def prolad_activate(model, step, total, sched="cosine"):
    modules = [(n,m) for n,m in model.named_modules() if hasattr(m,"activate") and hasattr(m,"lora_A")]
    n_tot = len(modules)
    progress = step / max(total - 1, 1)
    if sched == "cosine":
        n_act = max(1, int(n_tot * (1 - math.cos(progress * math.pi / 2))))
    else:  # immediate
        n_act = n_tot
    for i, (_, mod) in enumerate(modules):
        mod.activate(i < n_act)
    return n_act, n_tot

n_lora = inject_lora(base, r=8)
lora_p = sum(p.numel() for n,p in base.named_parameters() if "lora" in n)
print(f"  LoRA: {n_lora} modules, {lora_p/1e3:.1f}K params")

# ==== WikiText-103 Load ====
print("\nLoading WikiText-103...")
from datasets import load_dataset
ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="train")
# Filter non-empty lines, take 2000 paragraphs
texts = [t for t in ds["text"] if len(t.strip()) > 50][:2000]
print(f"  {len(texts)} paragraphs loaded")

# Tokenize and chunk
all_ids = []
for t in texts:
    ids = tok.encode(t, add_special_tokens=False, truncation=True, max_length=256)
    if len(ids) >= 32:
        all_ids.append(ids)
print(f"  {len(all_ids)} valid sequences")
random.shuffle(all_ids)

# ==== Training ====
def train_model(model, steps=200, sched="cosine"):
    opt = torch.optim.AdamW([p for n,p in model.named_parameters() if "lora" in n], lr=3e-3)
    model.train()
    losses = []
    for s in range(steps):
        ids = torch.tensor(all_ids[s % len(all_ids)], dtype=torch.long).unsqueeze(0).to(device)
        prolad_activate(model, s, steps, sched)
        loss = F.cross_entropy(model(ids).logits[0, :-1], ids[0, 1:])
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
        if s % 100 == 0:
            print(f"  Step {s:4d} | Loss: {loss.item():.4f} | Active: {sum(1 for m in model.modules() if hasattr(m,'active') and m.active)}/{sum(1 for m in model.modules() if hasattr(m,'active'))}")
    model.eval()
    return losses

def measure(model, ids, label="", set_active=None):
    """Measure PPL with LoRA on and off"""
    if set_active is not None:
        set_lora(model, set_active)
    with torch.no_grad():
        out = model(ids)
    return math.exp(F.cross_entropy(out.logits[0, :-1], ids[0, 1:]).item())

# ==== Phase 1: ProLAD (cosine) ====
print("\nPhase 1: ProLAD training (cosine schedule, 200 steps)...")
base.train()
l1 = train_model(base, 200, "cosine")
print(f"  Done: {l1[0]:.4f} -> {l1[-1]:.4f}")

prolad_state = {}
for n, p in base.named_parameters():
    if "lora" in n:
        prolad_state[n] = p.data.clone()

# ==== Phase 2: Baseline (immediate) ====
print("\nPhase 2: Baseline training (immediate schedule, 200 steps)...")
for n, p in base.named_parameters():
    if "lora_A" in n: nn.init.normal_(p, 0, 0.02)
    elif "lora_B" in n: nn.init.zeros_(p)

base.train()
l2 = train_model(base, 200, "immediate")
print(f"  Done: {l2[0]:.4f} -> {l2[-1]:.4f}")

baseline_state = {}
for n, p in base.named_parameters():
    if "lora" in n:
        baseline_state[n] = p.data.clone()

# ==== Evaluation at multiple lengths ====
print("\n" + "=" * 70)
print("EVALUATION")
print("=" * 70)

# Generate test text from WikiText (not in training set)
test_text = " ".join([
    t for t in texts[:500]  # overlap with training, that's fine for PPL
])[:8192]  # 8K buffer
test_ids = tok.encode(test_text, add_special_tokens=False)
print(f"  Test corpus: {len(test_ids)} tokens")

lengths = [64, 128, 256, 512, 1024, 2048, 4096]
max_len = min(4096, len(test_ids))
lengths = [l for l in lengths if l <= max_len]

def eval_state(model, state_dict, name):
    """Evaluate a saved state at all lengths."""
    with torch.no_grad():
        for n, p in model.named_parameters():
            if n in state_dict:
                p.data.copy_(state_dict[n])
    
    results = []
    for L in lengths:
        ids = torch.tensor(test_ids[:L], dtype=torch.long).unsqueeze(0).to(device)
        
        set_lora(model, False)
        ppl_off = measure(model, ids)
        
        set_lora(model, True)
        ppl_on = measure(model, ids)
        
        results.append({"len": L, "ppl_off": round(ppl_off, 4), "ppl_on": round(ppl_on, 4),
                        "ratio": round(ppl_on / max(ppl_off, 0.001), 2)})
        
        status = "✅" if ppl_off < ppl_on * 2 else "💥" if ppl_on > ppl_off * 10 else "⚠️"
        print(f"  {name:<10} L={L:>5} | PPL(off)={ppl_off:>10.2f} | PPL(on)={ppl_on:>10.2f} | Ratio={results[-1]['ratio']:>8.2f} {status}")
    
    return results

print(f"\n{'Method':<12} {'Len':>6} {'PPL(off)':>12} {'PPL(on)':>12} {'Ratio':>10}")
print(f"{'-'*12} {'-'*6} {'-'*12} {'-'*12} {'-'*10}")

prolad_results = eval_state(base, prolad_state, "ProLAD")
baseline_results = eval_state(base, baseline_state, "Baseline")

# ==== Summary ====
print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)
print(f"\n{'Len':>6} {'ProLAD off':>12} {'ProLAD on':>12} {'Basel off':>12} {'Basel on':>12}")
print(f"{'-'*6} {'-'*12} {'-'*12} {'-'*12} {'-'*12}")
for i, L in enumerate(lengths):
    p = prolad_results[i]
    b = baseline_results[i]
    print(f"{L:>6} {p['ppl_off']:>12.2f} {p['ppl_on']:>12.2f} {b['ppl_off']:>12.2f} {b['ppl_on']:>12.2f}")

# Save
output = {
    "config": {"model": MODEL, "train_steps": 200, "data": "WikiText-103", "rank": 8},
    "prolad": prolad_results,
    "baseline": baseline_results,
}
with open("/kaggle/working/wikitext_results.json", "w") as f:
    json.dump(output, f, indent=2)
print("\nResults saved to wikitext_results.json")

# Analysis
print("\n" + "=" * 70)
print("ANALYSIS")
print("=" * 70)

# Gap ratios
for name, res in [("ProLAD", prolad_results), ("Baseline", baseline_results)]:
    gaps = [r["ppl_on"] - r["ppl_off"] for r in res]
    ratios = [r["ratio"] for r in res]
    print(f"\n  {name}:")
    print(f"    Short gap (64t): {gaps[0]:.2f} | Long gap (4Kt): {gaps[-1]:.2f}")
    print(f"    Short ratio (64t): {ratios[0]:.2f}x | Long ratio (4Kt): {ratios[-1]:.2f}x")
    if len(gaps) >= 2:
        print(f"    Gap increase: {gaps[-1]/max(gaps[0],0.001):.1f}x")
        print(f"    Ratio increase: {ratios[-1]/max(ratios[0],0.001):.1f}x")

set_lora(base, False)
print("\nDone!")
